# Team Aggregation and Elo Ratings

In [ ]:
import json

import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "COMPETITION_DATA").exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "COMPETITION_DATA"
PROCESSED_DIR = REPO_ROOT / "processed_data"
PROCESSED_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)

In [ ]:
games_2022 = pd.read_csv(PROCESSED_DIR / "games_2022_processed.csv")
team_regions = pd.read_csv(DATA_DIR / "Team Region Groups - Sheet1.csv")
regional_games = pd.read_csv(DATA_DIR / "East Regional Games to predict - Sheet1.csv")

print(games_2022.shape)

In [12]:
team_agg = games_2022.groupby("team").agg({
    "win": ["mean", "sum"],
    "team_score": "mean",
    "opponent_team_score": "mean",
    "fg_pct_2": "mean",
    "fg_pct_3": "mean",
    "ft_pct": "mean",
    "AST": "mean",
    "BLK": "mean",
    "STL": "mean",
    "TOV": "mean",
    "TOV_team": "mean",
    "DREB": "mean",
    "OREB": "mean",
    "F_personal": "mean",
    "F_tech": "mean",
    "OT_length_min_tot": "mean",
    "largest_lead": "mean",
    "rest_days": "mean",
    "attendance": "mean",
    "tz_dif_H_E": "mean",
    "prev_game_dist": "mean",
    "travel_dist": "mean",
    "point_diff": "mean"
})

In [13]:
team_agg.columns = ["_".join(col).strip() for col in team_agg.columns.values]
team_agg = team_agg.reset_index()
team_agg.rename(columns={"win_mean": "win_percentage", "win_sum": "total_wins"}, inplace=True)
team_agg = team_agg.merge(team_regions, on="team", how="left")

## Elo ratings

In [14]:
all_teams = pd.concat([games_2022["team"], games_2022["team"]]).unique()
elo_ratings = {team: 1500 for team in all_teams}

In [15]:
def expected_score(rating_a, rating_b):
    """Calculate expected score for team A against team B."""
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

In [16]:
grouped_games = games_2022.groupby("game_id")
k = 20

In [17]:
for game_id, group in grouped_games:
    if group.shape[0] != 2:
        continue
    if "home" in group["home_away"].values and "away" in group["home_away"].values:
        home_row = group[group["home_away"] == "home"].iloc[0]
        away_row = group[group["home_away"] == "away"].iloc[0]
    else:
        home_row = group.iloc[0]
        away_row = group.iloc[1]

    home_team = home_row["team"]
    away_team = away_row["team"]
    home_score = home_row["team_score"]
    away_score = away_row["team_score"]

    outcome_home = 1 if home_score > away_score else 0
    outcome_away = 1 - outcome_home

    expected_home = expected_score(elo_ratings[home_team], elo_ratings[away_team])
    expected_away = expected_score(elo_ratings[away_team], elo_ratings[home_team])

    elo_ratings[home_team] += k * (outcome_home - expected_home)
    elo_ratings[away_team] += k * (outcome_away - expected_away)

In [18]:
team_agg["elo_rating"] = team_agg["team"].apply(lambda t: elo_ratings.get(t, 1500))

In [19]:
regional_games["elo_home"] = regional_games["team_home"].apply(lambda t: elo_ratings.get(t, 1500))
regional_games["elo_away"] = regional_games["team_away"].apply(lambda t: elo_ratings.get(t, 1500))
regional_games["expected_home"] = 1 / (1 + 10 ** ((regional_games["elo_away"] - regional_games["elo_home"]) / 400))
regional_games["predicted_winning_%_elo"] = regional_games["expected_home"]

## Combined ranking score

In [20]:
team_agg["combined_rank_score"] = team_agg["win_percentage"] * 0.5 + (team_agg["elo_rating"] / 3000) * 0.5
team_agg.sort_values("combined_rank_score", ascending=False, inplace=True)

In [22]:
team_agg

,team,win_percentage,total_wins,team_score_mean,opponent_team_score_mean,fg_pct_2_mean,fg_pct_3_mean,ft_pct_mean,AST_mean,BLK_mean,STL_mean,TOV_mean,TOV_team_mean,DREB_mean,OREB_mean,F_personal_mean,F_tech_mean,OT_length_min_tot_mean,largest_lead_mean,rest_days_mean,attendance_mean,tz_dif_H_E_mean,prev_game_dist_mean,travel_dist_mean,point_diff_mean,region,elo_rating,combined_rank_score
262,south_carolina_gamecocks,0.935484,29,0.441966,0.258667,0.476529,0.315578,0.676142,0.366487,0.712499,0.232975,0.308419,0.202061,0.563275,0.521169,0.312175,0.000000,0.010753,0.683199,0.394258,0.831776,0.467742,0.496808,0.408745,0.183298,North,1707.349408,0.752300
85,florida_gulf_coast_eagles,0.923077,24,0.485262,0.322070,0.574094,0.333584,0.643437,0.439103,0.474550,0.353276,0.224203,0.278662,0.420118,0.268029,0.357320,0.000000,0.000000,0.659617,0.410602,0.560688,0.500000,0.508339,0.371911,0.163192,North,1671.877632,0.740185
182,nc_state_wolfpack,0.906250,29,0.489486,0.308703,0.508094,0.367841,0.767493,0.368056,0.418930,0.269676,0.250762,0.068680,0.542468,0.375000,0.296371,0.018750,0.020833,0.652206,0.387569,0.747550,0.479167,0.403998,0.304096,0.180783,NaN,1710.561783,0.738219
278,stanford_cardinal,0.903226,28,0.465179,0.307205,0.500605,0.358444,0.673766,0.392473,0.631627,0.318996,0.268293,0.192523,0.491315,0.417339,0.386056,0.000000,0.000000,0.640675,0.378475,0.673757,0.483871,0.415306,0.420103,0.157974,West,1699.737450,0.734902
307,ucf_knights,0.888889,24,0.342679,0.219453,0.438113,0.307366,0.701903,0.349794,0.462390,0.417010,0.334237,0.268341,0.376068,0.429398,0.311828,0.007407,0.000000,0.580225,0.417590,0.611184,0.456790,0.575950,0.440928,0.123226,North,1668.665847,0.722555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,detroit_mercy_titans,0.037037,1,0.292835,0.418830,0.407079,0.262656,0.655214,0.276749,0.373759,0.334705,0.433604,0.181466,0.385565,0.358796,0.446834,0.014815,0.012346,0.208712,0.398821,0.416309,0.500000,0.417984,0.315842,-0.125995,NaN,1319.290953,0.238400
31,butler_bulldogs,0.035714,1,0.295060,0.503672,0.474366,0.261624,0.689546,0.288690,0.310307,0.182540,0.435540,0.296922,0.386447,0.290179,0.335253,0.000000,0.035714,0.203671,0.403987,0.516992,0.500000,0.413786,0.255909,-0.208611,NaN,1318.480270,0.237604
311,ul_monroe_warhawks,0.000000,0,0.304163,0.461342,0.386973,0.231866,0.669633,0.241162,0.394936,0.306397,0.385809,0.295437,0.334499,0.403409,0.486804,0.027273,0.015152,0.227693,0.426300,0.493391,0.492424,0.459042,0.315611,-0.157179,NaN,1331.890491,0.221982
64,delaware_state_hornets,0.000000,0,0.209856,0.490229,0.386449,0.233655,0.659957,0.188131,0.256635,0.161616,0.493348,0.485209,0.399767,0.291193,0.275660,0.045455,0.015152,0.159871,0.425030,0.449940,0.500000,0.511729,0.355291,-0.280374,NaN,1329.466526,0.221578


## Save the team table and ratings

In [ ]:
team_agg.to_csv(PROCESSED_DIR / "team_agg.csv", index=False)
regional_games.to_csv(PROCESSED_DIR / "regional_games_with_elo.csv", index=False)
with open(PROCESSED_DIR / "elo_ratings.json", "w") as f:
    json.dump(elo_ratings, f, indent=2)

print(f"Saved {len(team_agg)} teams and {len(elo_ratings)} Elo ratings to {PROCESSED_DIR}")